In [65]:
from SPARQLWrapper import SPARQLWrapper, JSON, CONSTRUCT, TURTLE
from rdflib.plugins.sparql import prepareQuery

import configparser
import rdflib
import pandas as pd

# since rdflib is not working properly with SPARQL construct, we use an alternative way using GraphDB for queries
# ... to use, please do the following steps: 
# 
# 1) change the flag below to True
# 2) prepare a SPARQL endpoint on the server and replace the sparql_endpoint name below
# 3) load both OCED ontology and resulted RDF file (in this case oced_ontology.ttl and 2013_small.ttl) to the prepared SPARQL endpoint


config = configparser.ConfigParser()
config.read("config.ini")

use_endpoint = config.get('config', 'use_endpoint')
sparql_endpoint = config.get('config', 'sparql_endpoint')

OCEDO_FIlENAME = config.get('config', 'oced_ontology')
OCEDD_FILENAME = config.get('config', 'ocedd_ontology')
INPUT_FILENAME =  config.get('config', 'trace_ttl')

OBJECT_OBJECT_MAP_FILENAME =  config.get('config', 'map_object-object')
EVENT_OBJECT_MAP_FILENAME =  config.get('config', 'map_event-object')

OUTPUT_FILENAME = config.get('config', 'enriched_trace_ttl')

In [66]:
# prepare rdflib graph
ontology_graph = rdflib.Graph()

ocedo = rdflib.Namespace('https://w3id.org/ocedo/core#')
ocedd = rdflib.Namespace('https://w3id.org/ocedo/domain#')
ocedr = rdflib.Namespace('https://w3id.org/ocedo/resource/')
ontology_graph.bind('ocedo', ocedo)
ontology_graph.bind('ocedd', ocedd)
ontology_graph.bind('ocedr', ocedr)

# load OCED ontology
ontology_graph.parse(OCEDO_FIlENAME, format="turtle")

# load input graph
input_graph = rdflib.Graph()
input_graph.bind('ocedo', ocedo)
input_graph.bind('ocedd', ocedd)
input_graph.bind('ocedr', ocedr)
input_graph.parse(INPUT_FILENAME, format="turtle")
input_graph += ontology_graph


In [67]:
# prepare construct query for event_object enhancement
# -- parameters needed: $object_type, $ocedd_class, $ocedd_relation
event_object_cq = """
prefix ocedo: <https://w3id.org/ocedo/core#>
prefix ocedd: <https://w3id.org/ocedo/domain#>
prefix ocedr: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?object a $ocedd_class .
    ?event $ocedd_relation ?object .
} 
WHERE {
    ?eo a ocedo:EventObject ;
    	ocedo:eo_event ?event ;
    	ocedo:eo_object ?object ;
    .
    ?object ocedo:instance_of_object "$object_type" .
}
"""

# prepare construct query for object_object enhancement
# -- parameters needed: $object1_class, $object2_class, $object1_type, $object2_type, $ocedd_relation
object_object_cq = """
prefix ocedo: <https://w3id.org/ocedo/core#>
prefix ocedd: <https://w3id.org/ocedo/domain#>
prefix ocedr: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?object1 a $object1_class .
    ?object2 a $object2_class .
    ?object1 $ocedd_relation ?object2 .
} 
WHERE {
    ?eo1 a ocedo:EventObject ;
    	ocedo:eo_event ?event ;
    	ocedo:eo_object ?object1 ;
    .
    ?eo2 a ocedo:EventObject ;
    	ocedo:eo_event ?event ;
    	ocedo:eo_object ?object2 ;
    .
    ?object1 ocedo:instance_of_object "$object1_type" .
    ?object2 ocedo:instance_of_object "$object2_type" .
}
"""

In [68]:
# prepare construct query for event_object enhancement
# -- parameters needed: $object_type, $ocedd_class, $ocedd_relation
event_object_ocedd_cq = """
prefix ocedo: <https://w3id.org/ocedo/core#>
prefix ocedd: <https://w3id.org/ocedo/domain#>
prefix ocedr: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix owl: <http://www.w3.org/2002/07/owl#>

CONSTRUCT {
    $ocedd_class a owl:Class ;
        rdfs:subClassOf ocedo:Object .
    $ocedd_relation a owl:ObjectProperty ;
        rdfs:domain ocedo:Event ;
        rdfs:range $ocedd_class ;
        rdfs:label "$ocedd_relation".
} WHERE {}
"""

# prepare construct query for object_object enhancement
# -- parameters needed: $object1_class, $object2_class, $object1_type, $object2_type, $ocedd_relation
object_object_ocedd_cq = """
prefix ocedo: <https://w3id.org/ocedo/core#>
prefix ocedd: <https://w3id.org/ocedo/domain#>
prefix ocedr: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix owl: <http://www.w3.org/2002/07/owl#>

CONSTRUCT {
    $object1_class a owl:Class ;
        rdfs:subClassOf ocedo:Object .
    $object2_class a owl:Class ;
        rdfs:subClassOf ocedo:Object .
    $ocedd_relation a owl:ObjectProperty ;
        rdfs:domain $object1_class ;
        rdfs:range $object2_class ;
        rdfs:label "$ocedd_relation".
} WHERE {}
"""

In [69]:

def build_event_object_cq(query, object_type, ocedd_class, ocedd_relation):
    cq = query
    cq = cq.replace("$object_type", object_type)
    cq = cq.replace("$ocedd_class", ocedd_class)
    cq = cq.replace("$ocedd_relation", ocedd_relation)
    return cq

def build_object_object_cq(query, object1_class, object2_class, object1_type, object2_type, ocedd_relation):
    cq = query
    cq = cq.replace("$object1_class", object1_class)
    cq = cq.replace("$object2_class", object2_class)
    cq = cq.replace("$object1_type", object1_type)
    cq = cq.replace("$object2_type", object2_type)
    cq = cq.replace("$ocedd_relation", ocedd_relation)
    return cq

def run_construct_query_endpoint(cq):
    sparql = SPARQLWrapper(sparql_endpoint)
    sparql.setQuery(cq)
    sparql.setReturnFormat(TURTLE)
    sparql.setMethod(CONSTRUCT)
    results = sparql.queryAndConvert()

    result_graph = rdflib.Graph()
    result_graph.parse(data=results, format='ttl')
    return result_graph

def run_construct_query_rdflib(cq):
    construct_query = prepareQuery(cq)
    result_graph = input_graph.query(construct_query).graph

    return result_graph


In [70]:
temp_data_result = rdflib.Graph()
temp_ocedd_result = rdflib.Graph()

df = pd.read_csv(EVENT_OBJECT_MAP_FILENAME)
for index, row in df.iterrows():
    o_type = row["object_type"]
    o_class = row["ocedd_class"]
    o_relation = row["ocedd_relation"]
    cq_string = build_event_object_cq(event_object_cq, o_type, o_class, o_relation)
    ocedo_cq_string = build_event_object_cq(event_object_ocedd_cq, o_type, o_class, o_relation)
    print(cq_string)
    if use_endpoint:
        temp_data_result += run_construct_query_endpoint(cq_string)
        temp_ocedd_result += run_construct_query_endpoint(ocedo_cq_string)
    else: 
        temp_data_result += run_construct_query_rdflib(cq_string)
        temp_ocedd_result += run_construct_query_rdflib(ocedo_cq_string)




prefix ocedo: <https://w3id.org/ocedo/core#>
prefix ocedd: <https://w3id.org/ocedo/domain#>
prefix ocedr: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?object a ocedd:Product .
    ?event ocedd:is_about_product ?object .
} 
WHERE {
    ?eo a ocedo:EventObject ;
    	ocedo:eo_event ?event ;
    	ocedo:eo_object ?object ;
    .
    ?object ocedo:instance_of_object "product" .
}


prefix ocedo: <https://w3id.org/ocedo/core#>
prefix ocedd: <https://w3id.org/ocedo/domain#>
prefix ocedr: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?object a ocedd:ProductFunction .
    ?event ocedd:is_about_function ?object .
} 
WHERE {
    ?eo a ocedo:EventObject ;
    	ocedo:eo_event ?event ;
    	ocedo:eo_object ?object ;
    .
    ?object ocedo:instance_of_object "org:role" .
}


prefix ocedo: <https://w3id.org/ocedo/core#>
prefix ocedd: <https://w3id.org/ocedo/domain#>
prefix ocedr: <

In [71]:
df = pd.read_csv(OBJECT_OBJECT_MAP_FILENAME)
for index, row in df.iterrows():
    object1_class = row["object1_class"]
    object2_class = row["object2_class"]
    object1_type = row["object1_type"]
    object2_type = row["object2_type"]
    ocedd_relation = row["ocedd_relation"]
    cq_string = build_object_object_cq(object_object_cq, object1_class, object2_class, object1_type, object2_type, ocedd_relation)
    ocedo_cq_string = build_object_object_cq(object_object_ocedd_cq, object1_class, object2_class, object1_type, object2_type, ocedd_relation)
    print(cq_string)
    if use_endpoint:
        temp_data_result += run_construct_query_endpoint(cq_string)
        temp_ocedd_result += run_construct_query_endpoint(ocedo_cq_string)
    else:
        temp_data_result += run_construct_query_rdflib(cq_string) # ==> not sure why, but it's really slow
        temp_ocedd_result += run_construct_query_rdflib(ocedo_cq_string)


prefix ocedo: <https://w3id.org/ocedo/core#>
prefix ocedd: <https://w3id.org/ocedo/domain#>
prefix ocedr: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?object1 a ocedd:TeamMember .
    ?object2 a ocedd:Location .
    ?object1 ocedd:located_in ?object2 .
} 
WHERE {
    ?eo1 a ocedo:EventObject ;
    	ocedo:eo_event ?event ;
    	ocedo:eo_object ?object1 ;
    .
    ?eo2 a ocedo:EventObject ;
    	ocedo:eo_event ?event ;
    	ocedo:eo_object ?object2 ;
    .
    ?object1 ocedo:instance_of_object "org:resource" .
    ?object2 ocedo:instance_of_object "resource country" .
}


prefix ocedo: <https://w3id.org/ocedo/core#>
prefix ocedd: <https://w3id.org/ocedo/domain#>
prefix ocedr: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?object1 a ocedd:TeamMember .
    ?object2 a ocedd:SupportTeam .
    ?object1 ocedd:part_of ?object2 .
} 
WHERE {
    ?eo1 a ocedo:EventObject ;
    

In [72]:
input_graph += temp_data_result
input_graph += temp_ocedd_result
input_graph.serialize(destination=OUTPUT_FILENAME, format='ttl')



<Graph identifier=Nc5c53dcd3d9945ee93b67230e56c9ae8 (<class 'rdflib.graph.Graph'>)>

In [73]:
ontology_graph +=temp_ocedd_result
ontology_graph.serialize(destination=OCEDD_FILENAME, format='ttl')

<Graph identifier=N456c814ee0fb4ea68cacfeb4328d231c (<class 'rdflib.graph.Graph'>)>